# Surrogate training

Trains the 5 -> 63 net on the generated surfaces. Metrics and loss curves go to MLflow (`mlflow ui`).

In [ ]:
import os, sys
os.chdir("..")   # run from the project root so paths match the scripts
sys.path.insert(0, ".")

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.train_surrogate import (run_training, load_model, evaluate,
                                 create_model, create_dataset, train_model)
from src.heston import PARAM_RANGES

data = np.load("data/heston_surfaces.npz")
moneyness, maturity = data["moneyness_grid"], data["maturity_grid"]
data["X_train"].shape, data["Y_train"].shape

## Train

`run_training` handles scaling, early stopping, MLflow logging and saving.

In [ ]:
metrics = run_training(epochs=300, patience=25)
metrics

## Loss curves

In [ ]:
import mlflow

run = mlflow.search_runs(experiment_names=["heston-surrogate"],
                         order_by=["start_time DESC"]).iloc[0]
client = mlflow.tracking.MlflowClient()
train = [(m.step, m.value) for m in client.get_metric_history(run.run_id, "train_mse")]
val = [(m.step, m.value) for m in client.get_metric_history(run.run_id, "val_mse")]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train), label="train")
plt.plot(*zip(*val), label="val")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("MSE (standardised)"); plt.legend(); plt.grid(alpha=0.3)
plt.title(f"RMSE {metrics['rmse_vol_pts']*100:.3f} vol points")

## Predicted vs true surface

In [ ]:
model, y_scaler = load_model("models/vol_surrogate.pt")

from src.train_surrogate import arrays_to_tensors, normalise_input
X_t, Y_t = arrays_to_tensors(data["X_test"], data["Y_test"])
with torch.no_grad():
    pred = y_scaler.inverse(model(normalise_input(X_t, PARAM_RANGES))).numpy()

i = 0
true_s = data["Y_test"][i].reshape(len(moneyness), len(maturity))
pred_s = pred[i].reshape(len(moneyness), len(maturity))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for T_idx, T in enumerate(maturity):
    axes[0].plot(moneyness, true_s[:, T_idx], marker=".", label=f"T={T}")
    axes[1].plot(moneyness, pred_s[:, T_idx], marker=".")
axes[0].set_title("COS pricer"); axes[1].set_title("surrogate")
axes[0].set_ylabel("implied vol"); axes[0].legend(fontsize=7)
for ax in axes: ax.set_xlabel("moneyness")
plt.tight_layout()

## Error distribution

In [ ]:
err = pred - data["Y_test"]
plt.figure(figsize=(7, 4))
plt.hist(err.ravel() * 100, bins=80)
plt.xlabel("error (vol points)"); plt.ylabel("count"); plt.grid(alpha=0.3)

worst = np.abs(err).max(axis=1)
print("worst surfaces:", np.argsort(-worst)[:5])
print("their params:
", data["X_test"][np.argsort(-worst)[:5]].round(3))